# Exercise 1 — Refactoring with Patterns (Easy)

---

## The story

A tiny coffee shop has one Python script that does everything. It works, but it is **messy**: object creation is tangled into `if/else` chains, and "add-ons" (milk, sugar, extra shot) are handled by stacking conditionals and string concatenation.

Your job is to refactor this single piece of code using **two patterns you already know**:

| Smell in the messy code | Pattern that fixes it |
| --- | --- |
| `if drink == "latte": ... elif ...` to build the drink object | **Factory** |
| `if add_milk: price += ...; name += ...` stacked add-ons | **Decorator** |

> **Goal.** Same behaviour, same output — but the conditionals for *creation* and for *add-ons* are gone, replaced by a factory and decorators.

Run the messy code first to see what it does, then refactor it in the cells below.

## 1. The messy code (read it, run it, don't keep it)

Notice the two problem areas:
1. `make_order` decides *what kind of drink* with an `if/elif` chain.
2. The add-ons (`milk`, `sugar`, `shot`) each mutate `name` and `price` with their own `if` block.

In [ ]:
# ---- MESSY CODE — this is what you will refactor ----

def make_order(drink, milk=False, sugar=False, shot=False):
    # creation by if/elif  -> smells like it wants a FACTORY
    if drink == "espresso":
        name = "Espresso"
        price = 2.00
    elif drink == "latte":
        name = "Latte"
        price = 3.00
    elif drink == "americano":
        name = "Americano"
        price = 2.50
    else:
        raise ValueError(f"unknown drink: {drink}")

    # add-ons by stacked if blocks -> smells like it wants DECORATORS
    if milk:
        name = name + " + milk"
        price = price + 0.50
    if sugar:
        name = name + " + sugar"
        price = price + 0.20
    if shot:
        name = name + " + extra shot"
        price = price + 0.80

    return f"{name}: ${price:.2f}"


print(make_order("latte", milk=True, sugar=True))
print(make_order("espresso", shot=True))
print(make_order("americano"))

## 2. Your task

Refactor the code above so that:

**Part A — Factory.** Replace the `if/elif` creation chain with a factory. Each drink should be its own small class (or object) carrying its `name` and `price`. A `drink_factory(kind)` function looks the kind up in a registry — **no `if/elif`**.

**Part B — Decorator.** Replace the three add-on `if` blocks with decorators that wrap a drink and adjust its `name` and `price`. Because decorators compose, you should be able to stack them (`Milk(Sugar(latte))`) instead of passing flags.

**Keep the output identical**, e.g. `"Latte + milk + sugar: $3.70"`.

### Suggested shape
```python
class Drink:            # common interface: .name() and .price()
    ...

# Part A: a registry-based factory  ->  drink_factory("latte")
# Part B: AddOn decorators           ->  Milk(Sugar(drink))
```

Write your solution in the next cell.

In [ ]:
# ---- YOUR SOLUTION ----

# Part A: Factory


# Part B: Decorators


# Demo — should match the messy code's output:
# print(...)  # Latte + milk + sugar: $3.70


## 3. One possible solution

Try it yourself first — then run the cell below to compare.

<details>
<summary>Reveal solution</summary>

The factory removes the creation `if/elif`; the decorators remove the add-on `if`s and compose freely.

</details>

In [ ]:
# ---- REFERENCE SOLUTION ----
from dataclasses import dataclass


# --- common interface ---
class Drink:
    def name(self) -> str: ...
    def price(self) -> float: ...

    def __str__(self) -> str:
        return f"{self.name()}: ${self.price():.2f}"


# --- concrete drinks ---
@dataclass
class _Base(Drink):
    _name: str
    _price: float
    def name(self):  return self._name
    def price(self): return self._price


# --- Part A: registry-based FACTORY (no if/elif) ---
_MENU = {
    "espresso":  lambda: _Base("Espresso", 2.00),
    "latte":     lambda: _Base("Latte", 3.00),
    "americano": lambda: _Base("Americano", 2.50),
}

def drink_factory(kind: str) -> Drink:
    try:
        return _MENU[kind]()
    except KeyError:
        raise ValueError(f"unknown drink: {kind}")


# --- Part B: DECORATORS (wrap a Drink, add behaviour, compose) ---
class AddOn(Drink):
    label = ""
    cost = 0.0
    def __init__(self, drink: Drink):
        self._wrapped = drink
    def name(self):  return f"{self._wrapped.name()} + {self.label}"
    def price(self): return self._wrapped.price() + self.cost

class Milk(AddOn):  label, cost = "milk", 0.50
class Sugar(AddOn): label, cost = "sugar", 0.20
class Shot(AddOn):  label, cost = "extra shot", 0.80


# --- demo: identical output to the messy code ---
print(Sugar(Milk(drink_factory("latte"))))   # Latte + milk + sugar: $3.70
print(Shot(drink_factory("espresso")))        # Espresso + extra shot: $2.80
print(drink_factory("americano"))             # Americano: $2.50

## 4. Takeaway

- **Factory** moved the "which class?" decision into a lookup table. Adding a new drink = one line in `_MENU`, no edits to existing code.
- **Decorator** turned add-ons into composable wrappers. `Milk(Sugar(...))` reads like the receipt, and you can stack them in any order.
- Notice the two patterns are **orthogonal**: the factory makes the base object, the decorators dress it up. They meet only at the shared `Drink` interface.

> Next exercise will combine more patterns on a harder piece of code.